## Generators & Augmentation testing

In this notebook we will try new augmentation algorithm over the cells

In [1]:
import pandas as pd
from composition.cells import CELLS

hungry = set(pd.read_parquet("data/composition/cell_order_sheet.parquet")["floor"])
audit = pd.DataFrame([
    {"cell": c.name, "hungry": c.name in hungry, "looks_like": c.looks_like}
    for c in CELLS
]).assign(written=lambda d: d["looks_like"].notna())

print(f"{audit['written'].sum()} of {len(audit)} cells have looks_like")
print(f"hungry cells still missing one: {(audit['hungry'] & ~audit['written']).sum()}")

audit[~audit["written"]].sort_values("hungry", ascending=False)[["cell", "hungry"]]

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


44 of 44 cells have looks_like
hungry cells still missing one: 0


,cell,hungry


In [5]:
audit[audit["written"]]

,cell,hungry,looks_like,written
17,symbol_pile_no_grammar,True,Two or more bare technical tokens sitting side...,True


In [2]:
import pandas as pd

from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS


In [3]:
paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
selection = pd.read_parquet(paths.data_dir / "composition/cell_selection.parquet").astype({"query_id": str})
parents = ParentPool(catalog, selection, {d.name: d for d in DATASETS})

In [3]:
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)
loop.plans(parents.available())

,cell,missing,parents,mints,unsatisfied,partial,action
0,legal_citation_canonical,400.0,347,"inject, stat_rewrite",,False,augment
1,datetime_token_present,399.0,184,"inject, stat_rewrite",,False,augment
2,single_token_char_blob,398.0,133143,stat_rewrite,,False,augment
3,business_temporal_reference,398.0,7,"inject, stat_rewrite",,False,augment
4,symbol_pile_no_grammar,398.0,90,"inject, stat_rewrite",,False,augment
5,bibliographic_catalog_identifier,398.0,332,"inject, stat_rewrite",,False,augment
6,bio_clinical_identifier,398.0,709,"inject, stat_rewrite",,False,augment
7,travel_transport_code,397.0,32,"inject, stat_rewrite",,False,augment
8,standards_compliance_lookup,394.0,57,"inject, stat_rewrite",,False,augment
9,boolean_operator_query,393.0,4304,operator_syntax_rewrite,,False,augment


In [4]:
CELL = "version_pinned_technical"          # two mints — exercises the pipeline
result, frame = loop.demand(CELL)

# grounded() attaches the gold document, but only when the plan CUTS — the cut
# has to keep that document answering, so it reads it (d53d)
parent = loop.grounded(result, frame.iloc[0])
parent[["dataset", "query_id", "query", "surfaces", "bank"]]

parents: dropped 2 row(s) with no query text


dataset               beir-nfcorpus
query_id                 PLAIN-3231
query       Meat & Multiple Myeloma
surfaces               (0.38-0.89,)
bank                 version_string
dtype: object

In [5]:
# d55: there is no longer ONE instruction per row. A cell becomes a sequence of
# calls — every addition in the first, the cut in a second — and each call's
# targets accumulate, so a later call cannot undo an earlier one.
from augmentation.dispatch import calls_for, targets_of
from composition.cells import CELLS_BY_NAME

for i, call in enumerate(calls_for(result, CELLS_BY_NAME[CELL], parent), 1):
    mints = [s.operator.declaration.operator for s in call.steps]
    print(f"{'=' * 25} CALL {i}  mints={mints}  cuts={call.cuts} {'=' * 25}")
    print(loop.brief(CELL, call.steps, parent))
    print("\nVERIFIED:", targets_of(call.verified, parent).model_dump_json(indent=1))
    print()

========================= CALL 1  mints=['inject']  cuts=False =========================
Weave the exact text '0.38-0.89' into the user's search query, on the same topic. The query may narrow — it no longer has to mean exactly what it meant. Insert them character for character, and do not alter them.

The finished query must look like this: A short technical query of three to nine words pinned to an exact dotted version string that must appear verbatim — the kind of precision embeddings actively blur, since adjacent versions differ by the whole point. Vary the job of the surrounding words: sometimes the version alone tags the answer, as in changelogs and release notes; sometimes the accompanying vocabulary — breaking change, migration, incompatibility — must select the right document among the many that mention the same version.


Add nothing beyond what is asked above: no other facts, names, numbers, dates, identifiers, greetings, or politeness phrases — anything extra changes the que

In [4]:
from augmentation.pool import GeneratedPool
from augmentation.qrels import AugmentationQrels

In [5]:
throwaway = AugmentationPaths(data_dir="/tmp/smoke")

loop = AugmentationLoop(
    selection, sheet_path=paths.cell_order_sheet, parents=parents,
    pool=GeneratedPool(throwaway), qrels=AugmentationQrels(throwaway),
)

In [ ]:

loop.run("version_pinned_technical", n=1)

parents: dropped 2 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:version_pinned_technical:   0%|          | 0/1 [00:02<?, ?row/s, attempted=1, dropped=1]

- PLAIN-3231: dropped — failed [version_string: at least 1 span(s) (measured 0.0), length_words >= 3.0 and <= 9.999999999 (measured 10.0)]
    before: 'Meat & Multiple Myeloma'
    tried:  'Meat multiple myeloma association 0.38-0.89 meta-analysis'


augment:version_pinned_technical:   0%|          | 0/1 [00:03<?, ?row/s, attempted=2, dropped=2]

- ca997dabb45f1928d6e4: dropped — failed [version_string: at least 1 span(s) (measured 0.0)]
    before: 'teen drama films from canada'
    tried:  'canadian teen drama films 3.86 release'


...

augment:version_pinned_technical:   0%|          | 0/1 [00:29<?, ?row/s, attempted=15, dropped=15]

- PLAIN-2332: dropped — failed [version_string: at least 1 span(s) (measured 0.0)]
    before: 'viral infections'
    tried:  'viral infection treatment guidelines 1.01'


augment:version_pinned_technical: 100%|██████████| 1/1 [00:32<00:00, 32.02s/row, attempted=16, dropped=15]

+ PLAIN-806 (2 attempt(s))
    before: 'caloric restriction'
    after:  'caloric restriction effects version 0.005'
version_pinned_technical: accepted 1/16 attempts (need 1, parents available 4,334, minting ['inject', 'stat_rewrite']) -> /tmp/smoke/augmentation/pool.parquet


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,credit_gate
0,aug-version_pinned_technical-PLAIN-806,caloric restriction effects version 0.005,version_pinned_technical,inject,doc_grounded,PLAIN-806,beir-nfcorpus,beir-nfcorpus,MED-1714,False,minted,2,coherence_gate


In [8]:
loop.run("symbol_pile_no_grammar", n=1)

parents: dropped 8 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:symbol_pile_no_grammar: 100%|██████████| 1/1 [00:05<00:00,  5.03s/row, attempted=1, dropped=0]

+ PLAIN-902 (1 attempt(s))
    before: 'chlorophyll'
    after:  'mL nCi chlorophyll'
symbol_pile_no_grammar: accepted 1/1 attempts (need 1, parents available 80, minting ['inject', 'stat_rewrite']) -> /tmp/smoke/augmentation/pool.parquet
  spend: 3 hops, 4,920 tokens, 7.3s wall (4.9s of it LLM) -> 3.0 hops/row, 4,920 tokens/row, 7.3s/row


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,hops,tokens,elapsed_s,credit_gate
0,aug-symbol_pile_no_grammar-PLAIN-902,mL nCi chlorophyll,symbol_pile_no_grammar,inject,doc_grounded,PLAIN-902,beir-nfcorpus,beir-nfcorpus,MED-4195,False,minted,1,3,4920,4.90825,coherence_gate


In [10]:
loop.run("legal_citation_canonical", n=5)

NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:legal_citation_canonical:  20%|██        | 1/5 [00:14<00:57, 14.31s/row, attempted=1, dropped=0]

+ 4373 (1 attempt(s))
    before: 'Are eviction cases first heard in circuit court? In the state of North Carolina'
    after:  '42 U.S.C. § 1983 North Carolina eviction magistrate'


augment:legal_citation_canonical:  40%|████      | 2/5 [00:21<00:29,  9.91s/row, attempted=2, dropped=0]

+ 6675 (1 attempt(s))
    before: 'In an eviction action, can a tenant use their status as a victim of domestic vio'
    after:  'eviction defense domestic violence Colorado 42 U.S.C.'


...

augment:legal_citation_canonical:  80%|████████  | 4/5 [00:45<00:11, 11.99s/row, attempted=4, dropped=0]

+ 6665 (1 attempt(s))
    before: 'In an eviction action, can a tenant rebut/raise the defense that the property is'
    after:  'Colorado tenant uninhabitable property defense eviction 42 U.S.C.'


augment:legal_citation_canonical: 100%|██████████| 5/5 [00:52<00:00, 10.57s/row, attempted=5, dropped=0]

+ 2267 (1 attempt(s))
    before: "Must a landlord accept a tenant's attempt to cure for substantial damage to prop"
    after:  'landlord tenant cure damage 42 U.S.C. § 1437d'
legal_citation_canonical: accepted 5/5 attempts (need 5, parents available 342, minting ['inject', 'stat_rewrite']) -> /tmp/smoke/augmentation/pool.parquet
  spend: 26 hops, 145,293 tokens, 53.6s wall (52.5s of it LLM) -> 5.2 hops/row, 29,059 tokens/row, 10.7s/row


,query_id,query,floor,operator,provenance,generated_from,parent_dataset,home_lane,grounding_doc_id,meaning_preserved,answer_key,attempts,hops,tokens,elapsed_s,credit_gate
0,aug-legal_citation_canonical-4373,42 U.S.C. § 1983 North Carolina eviction magis...,legal_citation_canonical,inject,doc_grounded,4373,crumb-legal-qa,crumb-legal-qa,f48c12cfed60d62eca0b,False,minted,1,7,48770,14.170700,coherence_gate
1,aug-legal_citation_canonical-6675,eviction defense domestic violence Colorado 42...,legal_citation_canonical,inject,doc_grounded,6675,crumb-legal-qa,crumb-legal-qa,73ddb8e3266cf6695101,False,minted,1,4,20580,6.788602,coherence_gate
2,aug-legal_citation_canonical-5376,42 U.S.C. § 401 et seq.,legal_citation_canonical,inject,doc_grounded,5376,crumb-legal-qa,crumb-legal-qa,58336d38423ff22e2458,False,minted,1,4,7475,6.757264,coherence_gate
3,aug-legal_citation_canonical-6665,Colorado tenant uninhabitable property defense...,legal_citation_canonical,inject,doc_grounded,6665,crumb-legal-qa,crumb-legal-qa,73ddb8e3266cf6695101,False,minted,1,7,47660,17.303073,coherence_gate
4,aug-legal_citation_canonical-2267,landlord tenant cure damage 42 U.S.C. § 1437d,legal_citation_canonical,inject,doc_grounded,2267,crumb-legal-qa,crumb-legal-qa,bffc74e582ddbe501600,False,minted,1,4,20808,7.465585,coherence_gate


In [6]:
from augmentation.campaign import AugmentationCampaign

campaign = AugmentationCampaign(loop)
campaign.plan()

parents: dropped 3 row(s) with no query text
parents: dropped 8 row(s) with no query text
parents: dropped 2 row(s) with no query text
parents: dropped 1 row(s) with no query text
parents: dropped 6 row(s) with no query text
parents: dropped 5 row(s) with no query text
parents: dropped 2 row(s) with no query text
campaign plan — 22 hungry floors, 639 target rows (>= 639 LLM calls):
                           floor  missing                operator              gate  action  target_rows
        legal_citation_canonical    400.0    inject, stat_rewrite    coherence_gate produce           20
          datetime_token_present    399.0    inject, stat_rewrite    coherence_gate produce           30
          single_token_char_blob    398.0            stat_rewrite declaration_audit produce           30
     business_temporal_reference    398.0    inject, stat_rewrite    coherence_gate produce           30
          symbol_pile_no_grammar    398.0    inject, stat_rewrite    coherence_gate produc

,floor,missing,operator,gate,action,target_rows
0,legal_citation_canonical,400.0,"inject, stat_rewrite",coherence_gate,produce,20
1,datetime_token_present,399.0,"inject, stat_rewrite",coherence_gate,produce,30
2,single_token_char_blob,398.0,stat_rewrite,declaration_audit,produce,30
3,business_temporal_reference,398.0,"inject, stat_rewrite",coherence_gate,produce,30
4,symbol_pile_no_grammar,398.0,"inject, stat_rewrite",coherence_gate,produce,27
5,bibliographic_catalog_identifier,398.0,"inject, stat_rewrite",coherence_gate,produce,30
6,bio_clinical_identifier,398.0,"inject, stat_rewrite",coherence_gate,produce,30
7,travel_transport_code,397.0,"inject, stat_rewrite",coherence_gate,produce,30
8,standards_compliance_lookup,394.0,"inject, stat_rewrite",coherence_gate,produce,30
9,boolean_operator_query,393.0,operator_syntax_rewrite,declaration_audit,produce,30


In [8]:
import contextlib
from pathlib import Path

log_path = Path("/tmp/campaign_run.log")
with log_path.open("w") as f, contextlib.redirect_stdout(f):
    summary = campaign.run()

print(f"done — {len(summary)} floors, full log at {log_path}")

                                                                                   
campaign:   0%|          | 0/22 [00:00<?, ?floor/s, floor=datetime_token_present]                      
                                                                                 
                                                                                                    

campaign:   0%|          | 0/22 [00:01<?, ?floor/s, floor=datetime_token_present]

                                                                                 
                                                                                                    

campaign:   0%|          | 0/22 [00:14<?, ?floor/s, floor=datetime_token_present]              

                                                                                 
                                                                                                    

campaign:   0%|          | 0/22 [00:25<?, ?floor/s, floor=datetime_token_present

KeyboardInterrupt: 

In [17]:
import pandas as pd
pool = pd.read_parquet("data/augmentation/pool.parquet")
pool["floor"].value_counts()   # rows per cell
pool[["query_id", "query", "floor", "credit_gate"]].tail(20)

,query_id,query,floor,credit_gate
2311,aug-code_symbol_named_in_prose-mbpp-q-273,How should I implement check_element to verify...,code_symbol_named_in_prose,coherence_gate
2312,aug-datetime_token_present-547,The Unseen 1981-06-05 old house,datetime_token_present,coherence_gate
2313,aug-datetime_token_present-176,Find divisibility problem 2015-11-11,datetime_token_present,coherence_gate
2314,aug-bibliographic_catalog_identifier-2021-2,aortic stenosis 1936-878X cardiac imaging,bibliographic_catalog_identifier,coherence_gate
2315,aug-bibliographic_catalog_identifier-2676,Does the law prohibit rental agreements from i...,bibliographic_catalog_identifier,coherence_gate
2316,aug-bibliographic_catalog_identifier-2923,A117.1 Missouri discrimination housing tenant,bibliographic_catalog_identifier,coherence_gate
2317,aug-bibliographic_catalog_identifier-1107,IC 22 tenant obligations falsification,bibliographic_catalog_identifier,coherence_gate
2318,aug-bibliographic_catalog_identifier-2021-30,M0 thyroid ablation remnant,bibliographic_catalog_identifier,coherence_gate
2319,aug-bibliographic_catalog_identifier-522,Let $f_{x} = c^{2x-6} \cdot f_{x-1} \cdot f_{x...,bibliographic_catalog_identifier,coherence_gate
2320,aug-bibliographic_catalog_identifier-2022-39,Osteoporosis diagnosis L5.5 vertebral compress...,bibliographic_catalog_identifier,coherence_gate
